# SpliceAI delta scores for the 1,130 synonymous alternatives of *GJB2* and *GJB6*

This notebook runs **only** the SpliceAI step (Cell 12b) and the Online Resource step (Cell 13) of `GJB2_GJB6_manuscript.ipynb`, starting from the table of alternatives produced by the full notebook (v1.1.0). The code of both cells is identical to the full notebook, so the scores are the same and the cache file `SpliceAI_GRCh38_1130.csv` written here can be reused by the full notebook.

**Files to upload when Cell A asks for them**
1. `TableS1_all_alternatives.csv` (GitHub repository, `results/` folder, v1.1.0)
2. `SpliceAI_CADDv1.7_GRCh38_952SNV.csv` (optional; used only to compare the computed SNV scores with the CADD v1.7 SpliceAI annotation)

Run all cells in order (Runtime > Run all). A GPU runtime is optional and only makes Cell 12b faster.

In [2]:
# == Cell A == Setup and table of alternatives (from the full notebook, v1.1.0) ======
import os, re, json, time, shutil
import numpy as np
import pandas as pd
import requests

WORKDIR = "./gjb_saturation"
os.makedirs(WORKDIR, exist_ok=True)

# Same constants as Cell 2 of the full notebook
CHROM       = "13"
GENE_STRAND = {"GJB2": -1, "GJB6": -1}
CDS_G_START = {"GJB2": 20189581, "GJB6": 20223480}
COMP_TBL    = str.maketrans("ACGTacgtNn-.", "TGCAtgcaNn-.")

def cds_to_genomic(gene, cds_off0):
    """0-based CDS offset (0 = c.1) -> GRCh38 genomic coordinate on chr13."""
    return (CDS_G_START[gene] - cds_off0 if GENE_STRAND[gene] == -1
            else CDS_G_START[gene] + cds_off0)

RUN_METADATA = {"manuscript_version": "MGG_submission_v1.2.0 (SpliceAI step only)", "software": {}}

NEEDED = ["TableS1_all_alternatives.csv"]
OPTIONAL = ["SpliceAI_CADDv1.7_GRCh38_952SNV.csv"]

# Helper to resolve files that might have been uploaded to /content/ with automatic suffix like (1).csv
for target_name in NEEDED + OPTIONAL:
    dest_path = os.path.join(WORKDIR, target_name)
    if not os.path.exists(dest_path):
        # Check if the exact file or a suffixed version exists in parent/current directory
        base_no_ext, ext = os.path.splitext(target_name)
        pattern = re.compile(rf"^{re.escape(base_no_ext)}(\s*\(\d+\))?{re.escape(ext)}$")
        for f in os.listdir("."):
            if pattern.match(f):
                shutil.move(f, dest_path)
                print(f"Moved and renamed {f} -> {dest_path}")
                break

missing = [f for f in NEEDED + OPTIONAL if not os.path.exists(os.path.join(WORKDIR, f))]
if missing:
    try:
        from google.colab import files
        print("Select the files:", ", ".join(missing))
        for name in files.upload():
            # Clean up the name from possible duplicate suffix
            clean_name = re.sub(r"\s*\(\d+\)", "", name)
            shutil.move(name, os.path.join(WORKDIR, clean_name))
    except ImportError:
        print("Copy the files into", WORKDIR)

for f in NEEDED:
    assert os.path.exists(os.path.join(WORKDIR, f)), f"{f} is required"

atlas = pd.read_csv(os.path.join(WORKDIR, "TableS1_all_alternatives.csv"))
atlas = atlas.drop(columns=[c for c in atlas.columns if c.startswith("SpliceAI_")])
needed_cols = ["gene", "variant_c", "variant_p", "codon_index", "wt_codon", "alt_codon", "snv_reachable",
               "g_pos", "g_ref", "g_alt", "in_clinvar", "clinsig", "wt_wCAI", "alt_wCAI"]
assert len(atlas) == 1130 and all(c in atlas.columns for c in needed_cols), "unexpected table"
atlas["in_clinvar"] = atlas["in_clinvar"].astype(bool)
print(f"Table of alternatives: {len(atlas)} rows, {atlas.shape[1]} columns; "
      f"SNVs {int(atlas.snv_reachable.astype(bool).sum())}, MNVs {int((~atlas.snv_reachable.astype(bool)).sum())}")

Moved and renamed TableS1_all_alternatives.csv -> ./gjb_saturation/TableS1_all_alternatives.csv
Moved and renamed SpliceAI_CADDv1.7_GRCh38_952SNV.csv -> ./gjb_saturation/SpliceAI_CADDv1.7_GRCh38_952SNV.csv
Table of alternatives: 1130 rows, 44 columns; SNVs 952, MNVs 178


In [3]:
# == Cell 12b == SpliceAI Delta Scores for All 1,130 Synonymous Alternatives (2.4, 3.5) ==
# SpliceAI v1.3.1 (Jaganathan et al. 2019) with its five published models and its bundled GRCh38
# gene annotation (GENCODE V24 canonical transcripts), using the command-line defaults: maximum
# distance 50 nt (-D 50) and raw (unmasked) delta scores (-M 0); masked scores are also stored.
# The SpliceAI command-line tool skips substitutions in which both alleles are longer than one
# nucleotide. Its scoring function (spliceai.utils.get_delta_scores) is therefore reproduced below
# with that restriction lifted for equal-length substitutions (MNVs), where the alternative sequence
# is the reference sequence carrying the substituted block, exactly as for SNVs. For SNVs the
# function is identical to the official implementation. Reference sequence: GRCh38 chromosome 13
# from Ensembl REST. Scores are cached in SpliceAI_GRCh38_1130.csv; delete that file to recompute.
import importlib.util

SAI_CACHE = os.path.join(WORKDIR, "SpliceAI_GRCh38_1130.csv")
SAI_DIST, SAI_COV = 50, 101
SAI_WID = 10000 + SAI_COV
DS_COLS = ["SpliceAI_DS_AG", "SpliceAI_DS_AL", "SpliceAI_DS_DG", "SpliceAI_DS_DL"]
DP_COLS = ["SpliceAI_DP_AG", "SpliceAI_DP_AL", "SpliceAI_DP_DG", "SpliceAI_DP_DL"]
DSM_COLS = [c + "_masked" for c in DS_COLS]

# 1. Genomic description of every alternative (block from the first to the last changed base)
def genomic_block(gene, codon_index, wt, alt):
    q = [i for i in range(3) if wt[i] != alt[i]]
    c_first = (codon_index - 1) * 3 + min(q)             # 0-based CDS offsets
    c_last = (codon_index - 1) * 3 + max(q)
    ref_c, alt_c = wt[min(q):max(q) + 1], alt[min(q):max(q) + 1]
    if GENE_STRAND[gene] == -1:
        return cds_to_genomic(gene, c_last), ref_c.translate(COMP_TBL)[::-1], alt_c.translate(COMP_TBL)[::-1]
    return cds_to_genomic(gene, c_first), ref_c, alt_c

blocks = pd.DataFrame([dict(gene=r.gene, variant_c=r.variant_c,
                            **dict(zip(["blk_pos", "blk_ref", "blk_alt"],
                                       genomic_block(r.gene, int(r.codon_index), r.wt_codon, r.alt_codon))))
                       for r in atlas.itertuples()])
assert len(blocks) == 1130 and not blocks.duplicated(["gene", "variant_c"]).any()
_snv = atlas[atlas["snv_reachable"].astype(bool)].merge(blocks, on=["gene", "variant_c"])
assert (_snv.blk_pos == _snv.g_pos).all() and (_snv.blk_ref == _snv.g_ref).all() and (_snv.blk_alt == _snv.g_alt).all(), \
    "genomic block does not match the SNV coordinates"

if os.path.exists(SAI_CACHE):
    sai = pd.read_csv(SAI_CACHE)
    print(f"Loaded cached SpliceAI scores: {SAI_CACHE} ({len(sai)} alternatives)")
else:
    # 2. SpliceAI models and annotation (package files only; Colab's TensorFlow/Keras is used)
    if importlib.util.find_spec("spliceai") is None:
        !pip -q install --no-deps "spliceai==1.3.1"
    import keras
    keras.utils.disable_interactive_logging()
    SAI_DIR = os.path.dirname(importlib.util.find_spec("spliceai").origin)
    SAI_MODELS = [keras.models.load_model(os.path.join(SAI_DIR, "models", f"spliceai{i}.h5"), compile=False)
                  for i in range(1, 6)]
    ann = pd.read_csv(os.path.join(SAI_DIR, "annotations", "grch38.txt"), sep="\t", dtype={"CHROM": str})
    ann = ann[ann["CHROM"] == CHROM].copy()
    ann["tx_s"] = ann["TX_START"] + 1                      # as in spliceai.utils.Annotator
    ann["tx_e"] = ann["TX_END"]
    ann["ex_s"] = [np.asarray(list(map(int, x.split(",")[:-1]))) + 1 for x in ann["EXON_START"]]
    ann["ex_e"] = [np.asarray(list(map(int, x.split(",")[:-1]))) for x in ann["EXON_END"]]

    # 3. GRCh38 reference sequence covering every scoring window
    lo = int(blocks.blk_pos.min()) - SAI_WID
    hi = int(blocks.blk_pos.max()) + SAI_WID
    def fetch_region(lo, hi):
        """GRCh38 chr13:lo-hi (1-based, inclusive), plus strand: Ensembl REST, or the UCSC API if Ensembl is unavailable."""
        for attempt in range(3):
            try:
                r = requests.get(f"https://rest.ensembl.org/sequence/region/human/{CHROM}:{lo}..{hi}:1",
                                 headers={"Content-Type": "text/plain"}, timeout=120)
                if r.status_code == 200:
                    return r.text.strip().upper(), "Ensembl REST"
                print(f"Ensembl REST returned {r.status_code}")
            except requests.RequestException as e:
                print(f"Ensembl REST error: {e}")
            time.sleep(3 * (attempt + 1))
        # UCSC API: parameters separated by semicolons; start is 0-based, end is 1-based
        r = requests.get(f"https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom=chr{CHROM};start={lo - 1};end={hi}",
                         timeout=120)
        r.raise_for_status()
        return r.json()["dna"].strip().upper(), "UCSC API (hg38)"
    REGION, REGION_SOURCE = fetch_region(lo, hi)
    assert len(REGION) == hi - lo + 1, "unexpected region length"
    print(f"GRCh38 chr{CHROM}:{lo:,}-{hi:,} retrieved from {REGION_SOURCE}")
    def ref_slice(a, b):                                    # 1-based inclusive genomic interval
        return REGION[a - lo:b - lo + 1]

    # 4. Input sequences, built exactly as in spliceai.utils.get_delta_scores
    _MAP = np.asarray([[0, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=np.float32)
    def one_hot(s):
        s = s.upper().replace("A", "\x01").replace("C", "\x02").replace("G", "\x03").replace("T", "\x04").replace("N", "\x00")
        return _MAP[np.frombuffer(s.encode("latin-1"), np.int8) % 5]

    recs, ref_inputs, alt_inputs = [], {}, []
    for b in blocks.itertuples():
        pos, ref, alt = int(b.blk_pos), b.blk_ref, b.blk_alt
        g = ann[(ann.tx_s <= pos) & (pos <= ann.tx_e)]
        assert len(g) == 1 and g.iloc[0]["#NAME"] == b.gene, f"annotation overlap problem at {pos}"
        g = g.iloc[0]
        seq = ref_slice(pos - SAI_WID // 2, pos + SAI_WID // 2)
        assert len(seq) == SAI_WID and seq[SAI_WID // 2:SAI_WID // 2 + len(ref)] == ref, \
            f"reference mismatch at {b.gene} {b.variant_c}"
        dist_tx_start, dist_tx_end = g.tx_s - pos, g.tx_e - pos
        dist_exon_bdry = min(np.union1d(g.ex_s, g.ex_e) - pos, key=abs)
        pad = [max(SAI_WID // 2 + dist_tx_start, 0), max(SAI_WID // 2 - dist_tx_end, 0)]
        x_ref = "N" * pad[0] + seq[pad[0]:SAI_WID - pad[1]] + "N" * pad[1]
        x_alt = x_ref[:SAI_WID // 2] + alt + x_ref[SAI_WID // 2 + len(ref):]
        key = (pos, g.STRAND)
        ref_inputs.setdefault(key, x_ref)
        assert ref_inputs[key] == x_ref
        recs.append(dict(gene=b.gene, variant_c=b.variant_c, key=key, strand=g.STRAND,
                         dist_exon_bdry=int(dist_exon_bdry), alt_idx=len(alt_inputs)))
        alt_inputs.append(x_alt)

    # 5. Batched prediction with the five models (minus-strand inputs are reverse-complemented)
    def predict(strings, strands, batch=32):
        out = []
        for i in range(0, len(strings), batch):
            x = np.stack([one_hot(s) for s in strings[i:i + batch]])
            st = strands[i:i + batch]
            for k, s in enumerate(st):
                if s == "-":
                    x[k] = x[k, ::-1, ::-1]
            y = np.mean([m.predict(x, verbose=0) for m in SAI_MODELS], axis=0)
            for k, s in enumerate(st):
                out.append(y[k, ::-1] if s == "-" else y[k])
        return out

    t0 = time.time()
    ref_keys = list(ref_inputs)
    y_ref_all = dict(zip(ref_keys, predict([ref_inputs[k] for k in ref_keys], [k[1] for k in ref_keys])))
    y_alt_all = predict(alt_inputs, [r["strand"] for r in recs])
    print(f"SpliceAI predictions: {len(ref_keys)} reference and {len(alt_inputs)} alternative sequences "
          f"in {time.time() - t0:.0f} s")

    # 6. Delta scores (raw and masked) and positions, as in spliceai.utils.get_delta_scores
    rows = []
    for rec in recs:
        y = np.stack([y_ref_all[rec["key"]], y_alt_all[rec["alt_idx"]]])
        idx = [(y[1, :, 1] - y[0, :, 1]).argmax(), (y[0, :, 1] - y[1, :, 1]).argmax(),
               (y[1, :, 2] - y[0, :, 2]).argmax(), (y[0, :, 2] - y[1, :, 2]).argmax()]
        raw = [y[1, idx[0], 1] - y[0, idx[0], 1], y[0, idx[1], 1] - y[1, idx[1], 1],
               y[1, idx[2], 2] - y[0, idx[2], 2], y[0, idx[3], 2] - y[1, idx[3], 2]]
        dp = [int(i - SAI_COV // 2) for i in idx]
        d = rec["dist_exon_bdry"]
        msk = [dp[0] == d, dp[1] != d, dp[2] == d, dp[3] != d]  # gain at annotated site / loss at unannotated site
        row = dict(gene=rec["gene"], variant_c=rec["variant_c"])
        row.update({c: round(float(v), 2) + 0.0 for c, v in zip(DS_COLS, raw)})
        row.update({c: (0.0 if m else round(float(v), 2) + 0.0) for c, v, m in zip(DSM_COLS, raw, msk)})
        row.update(dict(zip(DP_COLS, dp)))
        rows.append(row)
    sai = blocks.merge(pd.DataFrame(rows), on=["gene", "variant_c"])
    sai.to_csv(SAI_CACHE, index=False)
    print(f"Saved {SAI_CACHE}")

assert len(sai) == 1130

# 7. Validation against the precomputed SpliceAI scores reported by CADD v1.7 for the 952 SNVs (optional)
CADD_SAI = os.path.join(WORKDIR, "SpliceAI_CADDv1.7_GRCh38_952SNV.csv")
if os.path.exists(CADD_SAI):
    cadd = pd.read_csv(CADD_SAI).rename(columns={"g_pos": "blk_pos", "g_ref": "blk_ref", "g_alt": "blk_alt"})
    cmp_ = sai.merge(cadd, on=["blk_pos", "blk_ref", "blk_alt"], suffixes=("", "_cadd"))
    assert len(cmp_) == 952
    for label, cols in (("raw", DS_COLS), ("masked", DSM_COLS)):
        diff = np.abs(cmp_[cols].to_numpy() - cmp_[[c + "_cadd" for c in DS_COLS]].to_numpy())
        print(f"Agreement with CADD v1.7 SpliceAI scores ({label}): identical in "
              f"{int((diff.max(axis=1) < 0.005).sum())}/952 SNVs, maximum absolute difference {diff.max():.2f}")

# 8. Merge onto the table of alternatives and save
atlas = atlas.drop(columns=[c for c in DS_COLS + DP_COLS + ["SpliceAI_DS_max"] if c in atlas.columns])
atlas = atlas.merge(sai[["gene", "variant_c"] + DS_COLS + DP_COLS], on=["gene", "variant_c"], how="left")
atlas["SpliceAI_DS_max"] = atlas[DS_COLS].max(axis=1)
assert len(atlas) == 1130 and atlas["SpliceAI_DS_max"].notna().all()
atlas.to_csv(os.path.join(WORKDIR, "TableS1_all_alternatives.csv"), index=False)
cv_mapped = atlas[atlas["in_clinvar"]].copy()
cv_mapped.to_csv(os.path.join(WORKDIR, "ClinVar_Synonymous_Atlas_Mapped.csv"), index=False)

# 9. Summary reported in Section 3.5
is_snv = atlas["snv_reachable"].astype(bool)
print("\n=== 3.5 SpliceAI Delta Scores of the 1,130 Synonymous Alternatives ===")
for label, sub in (("All alternatives", atlas), ("SNVs", atlas[is_snv]), ("MNVs", atlas[~is_snv])):
    n = len(sub)
    print(f"{label}: n = {n}; maximum delta score = 0 in {int((sub.SpliceAI_DS_max == 0).sum())} "
          f"({100 * (sub.SpliceAI_DS_max == 0).mean():.1f}%), < 0.10 in {int((sub.SpliceAI_DS_max < 0.10).sum())} "
          f"({100 * (sub.SpliceAI_DS_max < 0.10).mean():.1f}%), >= 0.20 in {int((sub.SpliceAI_DS_max >= 0.20).sum())}, "
          f">= 0.50 in {int((sub.SpliceAI_DS_max >= 0.50).sum())}")
for gene in ("GJB2", "GJB6"):
    g = atlas[atlas.gene == gene]
    print(f"  {gene}: {len(g)} alternatives, score > 0 in {int((g.SpliceAI_DS_max > 0).sum())}, maximum {g.SpliceAI_DS_max.max():.2f}")
for c in DS_COLS:
    print(f"  {c}: > 0 in {int((atlas[c] > 0).sum())}, maximum {atlas[c].max():.2f}")

print("\nAlternatives with a maximum delta score >= 0.10 (DP = position of the predicted change relative to the variant, nt):")
top = atlas[atlas.SpliceAI_DS_max >= 0.10].sort_values("SpliceAI_DS_max", ascending=False)
print(top[["gene", "variant_c", "variant_p", "snv_reachable"] + DS_COLS + DP_COLS + ["clinsig"]].to_string(index=False))

cvs = atlas[atlas["in_clinvar"]]
unres = cvs[cvs["clinsig"].str.contains("Uncertain|Conflicting", case=False, na=False)]
print(f"\nClinVar-listed SNVs: {len(cvs)}, maximum delta score {cvs.SpliceAI_DS_max.max():.2f}, "
      f">= 0.10 in {int((cvs.SpliceAI_DS_max >= 0.10).sum())}")
print(f"Uncertain or conflicting ClinVar SNVs: {len(unres)}, maximum delta score {unres.SpliceAI_DS_max.max():.2f}")
RUN_METADATA["software"]["SpliceAI"] = "1.3.1 (models and GRCh38 annotation; -D 50, raw scores)"
with open(os.path.join(WORKDIR, "run_metadata_manuscript.json"), "w") as fh:
    json.dump(RUN_METADATA, fh, indent=2)

Ensembl REST error: HTTPSConnectionPool(host='rest.ensembl.org', port=443): Max retries exceeded with url: /sequence/region/human/13:20178803..20233576:1 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7cfae302ef90>, 'Connection to rest.ensembl.org timed out. (connect timeout=120)'))
Ensembl REST returned 403
Ensembl REST returned 403
GRCh38 chr13:20,178,803-20,233,576 retrieved from UCSC API (hg38)


SpliceAI predictions: 533 reference and 1130 alternative sequences in 4481 s
Saved ./gjb_saturation/SpliceAI_GRCh38_1130.csv
Agreement with CADD v1.7 SpliceAI scores (raw): identical in 928/952 SNVs, maximum absolute difference 0.03
Agreement with CADD v1.7 SpliceAI scores (masked): identical in 952/952 SNVs, maximum absolute difference 0.00

=== 3.5 SpliceAI Delta Scores of the 1,130 Synonymous Alternatives ===
All alternatives: n = 1130; maximum delta score = 0 in 1026 (90.8%), < 0.10 in 1123 (99.4%), >= 0.20 in 4, >= 0.50 in 3
SNVs: n = 952; maximum delta score = 0 in 869 (91.3%), < 0.10 in 948 (99.6%), >= 0.20 in 1, >= 0.50 in 1
MNVs: n = 178; maximum delta score = 0 in 157 (88.2%), < 0.10 in 175 (98.3%), >= 0.20 in 3, >= 0.50 in 2
  GJB2: 526 alternatives, score > 0 in 58, maximum 0.77
  GJB6: 604 alternatives, score > 0 in 46, maximum 0.85
  SpliceAI_DS_AG: > 0 in 61, maximum 0.85
  SpliceAI_DS_AL: > 0 in 47, maximum 0.04
  SpliceAI_DS_DG: > 0 in 9, maximum 0.04
  SpliceAI_DS_DL:

In [4]:
# == Cell 13 == Electronic Supplementary Material: Online Resources 1 and 2 ========
# Springer requires supplementary files to be named consecutively (ESM_1, ESM_2) and
# each file to carry the article title, journal, author names and the affiliation and
# e-mail address of the corresponding author.
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment

ARTICLE_TITLE = ("Codon Architecture of the Paralogous Connexin Genes GJB2 and GJB6 and a "
                 "Complete Map of Their Synonymous Alternatives at the DFNB1 Locus")
JOURNAL = "Molecular Genetics and Genomics"
AUTHORS = "Seda Nur Homurlu, Oğuz Altungöz"
CORRESPONDING = ("Corresponding author: Oğuz Altungöz, Dokuz Eylül University, Faculty of Medicine, "
                 "Department of Medical Biology and Genetics, 35340 Izmir, Türkiye; "
                 "oguz.altungoz@deu.edu.tr")

COLUMN_DESCRIPTIONS = {
    "gene": "Gene symbol",
    "transcript": "RefSeq transcript",
    "codon_index": "Codon position in the coding sequence (1 = initiator Met)",
    "aa": "Encoded amino acid (one-letter code)",
    "wt_codon": "Reference codon",
    "alt_codon": "Synonymous alternative codon",
    "alt_row": "Index of the alternative at this position (row in Fig. 4)",
    "variant_c": "HGVS coding DNA description",
    "variant_p": "HGVS protein description",
    "n_nt_changes": "Number of nucleotides changed",
    "snv_reachable": "TRUE = single-nucleotide variant (SNV); FALSE = multinucleotide variant (MNV)",
    "c_pos": "Coding DNA position of the changed nucleotide (SNVs only)",
    "c_ref": "Reference nucleotide, coding strand (SNVs only)",
    "c_alt": "Alternative nucleotide, coding strand (SNVs only)",
    "g_pos": "GRCh38 chromosome 13 position (SNVs only)",
    "g_ref": "Reference nucleotide, genomic plus strand (SNVs only)",
    "g_alt": "Alternative nucleotide, genomic plus strand (SNVs only)",
    "wt_RSCU_gene": "Gene-specific RSCU of the reference codon",
    "alt_RSCU_gene": "Gene-specific RSCU of the alternative codon",
    "dRSCU_gene": "ΔRSCU (alternative minus reference)",
    "wt_wCAI": "Genome-wide codon adaptation weight (wCAI) of the reference codon",
    "alt_wCAI": "wCAI of the alternative codon",
    "delta_cai_weight": "ΔwCAI (alternative minus reference)",
    "dCAI_transcript": "Change in gene CAI caused by the single substitution",
    "wt_wtAI": "tRNA adaptation weight (wtAI) of the reference codon",
    "alt_wtAI": "wtAI of the alternative codon",
    "delta_tai_weight": "ΔwtAI (alternative minus reference)",
    "WT_Gens_codon_40": "Ensemble free energy of the 81-nt reference window (kcal/mol)",
    "ALT_Gens_codon_40": "Ensemble free energy of the 81-nt alternative window (kcal/mol)",
    "dGens_codon_40": "ΔGens (alternative minus reference, kcal/mol; positive = destabilization)",
    "WT_MFE_codon_40": "Minimum free energy of the 81-nt reference window (kcal/mol)",
    "ALT_MFE_codon_40": "Minimum free energy of the 81-nt alternative window (kcal/mol)",
    "dMFE_codon_40": "ΔMFE (alternative minus reference, kcal/mol)",
    "hgvs_c": "ClinVar variant name",
    "clinvar_id": "ClinVar VariationID",
    "clinsig": "ClinVar germline classification",
    "review": "ClinVar review status",
    "in_clinvar": "TRUE = variant reported in ClinVar",
    "ancestral_codon": "Inferred human–chimpanzee ancestral codon",
    "ancestral_source": "Source of the ancestral codon (Ensembl EPO or parsimony)",
    "reference_codon_derived": "TRUE = human reference codon differs from the ancestral codon",
    "primates_with_alt_codon": "Number of the 23 non-human primates carrying the alternative codon",
    "species_with_alt_codon": "Primates carrying the alternative codon",
    "evolutionary_class": "Evolutionary classification of the alternative",
    "wt_codon_rarely_used": "TRUE = reference codon is rarely used (wCAI < 0.5)",
    "alt_codon_rarely_used": "TRUE = alternative codon is rarely used (wCAI < 0.5)",
    "SpliceAI_DS_AG": "SpliceAI delta score, acceptor gain (SpliceAI v1.3.1, maximum distance 50 nt, raw score)",
    "SpliceAI_DS_AL": "SpliceAI delta score, acceptor loss (SpliceAI v1.3.1, maximum distance 50 nt, raw score)",
    "SpliceAI_DS_DG": "SpliceAI delta score, donor gain (SpliceAI v1.3.1, maximum distance 50 nt, raw score)",
    "SpliceAI_DS_DL": "SpliceAI delta score, donor loss (SpliceAI v1.3.1, maximum distance 50 nt, raw score)",
    "SpliceAI_DP_AG": "Position of the predicted acceptor gain relative to the variant (nt; genomic plus-strand orientation, so for these minus-strand genes a positive value lies 5′ of the variant in the transcript)",
    "SpliceAI_DP_AL": "Position of the predicted acceptor loss relative to the variant (nt; genomic plus-strand orientation, so for these minus-strand genes a positive value lies 5′ of the variant in the transcript)",
    "SpliceAI_DP_DG": "Position of the predicted donor gain relative to the variant (nt; genomic plus-strand orientation, so for these minus-strand genes a positive value lies 5′ of the variant in the transcript)",
    "SpliceAI_DP_DL": "Position of the predicted donor loss relative to the variant (nt; genomic plus-strand orientation, so for these minus-strand genes a positive value lies 5′ of the variant in the transcript)",
    "SpliceAI_DS_max": "Maximum of the four SpliceAI delta scores",
}

def write_esm(df, path, resource_no, description):
    missing = [c for c in df.columns if c not in COLUMN_DESCRIPTIONS]
    assert not missing, f"undocumented columns: {missing}"
    header = [f"Online Resource {resource_no}", ARTICLE_TITLE, JOURNAL, AUTHORS, CORRESPONDING, description]
    start = len(header) + 1
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        df.to_excel(xw, sheet_name=f"Online Resource {resource_no}", index=False, startrow=start)
        pd.DataFrame({"Column": list(df.columns),
                      "Description": [COLUMN_DESCRIPTIONS[c] for c in df.columns]}
                     ).to_excel(xw, sheet_name="Column descriptions", index=False)
    wb = load_workbook(path)
    ws = wb[f"Online Resource {resource_no}"]
    for i, line in enumerate(header, 1):
        ws.cell(row=i, column=1, value=line).font = Font(bold=(i == 1))
    for cell in ws[start + 1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = ws.cell(row=start + 2, column=1)
    for col in ws.iter_cols(min_row=start + 1, max_row=start + 1):
        ws.column_dimensions[col[0].column_letter].width = 16
    ws2 = wb["Column descriptions"]
    for cell in ws2[1]:
        cell.font = Font(bold=True)
    ws2.column_dimensions["A"].width = 26
    ws2.column_dimensions["B"].width = 90
    wb.save(path)
    print(f"Saved {os.path.basename(path)}: {len(df)} rows x {df.shape[1]} columns")

# Online Resource 1 (ESM_1.xlsx): all 1,130 synonymous alternatives
esm1 = atlas.copy()
esm1["wt_codon_rarely_used"]  = esm1["wt_wCAI"] < 0.5
esm1["alt_codon_rarely_used"] = esm1["alt_wCAI"] < 0.5
esm1 = esm1.sort_values(["gene", "codon_index", "alt_row"]).reset_index(drop=True)
assert len(esm1) == 1130
write_esm(esm1, os.path.join(WORKDIR, "ESM_1.xlsx"), 1,
          "Complete enumeration of all 1130 single-codon synonymous alternatives in GJB2 and GJB6, "
          "with their ΔRSCU, ΔwCAI, ΔwtAI and ΔGens values, SpliceAI delta scores, "
          "evolutionary classification and ClinVar annotation.")

# Online Resource 2 (ESM_2.xlsx): ClinVar synonymous variants with uncertain or conflicting classifications
ESM2_COLS = ["gene", "transcript", "codon_index", "aa", "variant_c", "variant_p", "clinvar_id", "clinsig",
             "review", "wt_codon", "alt_codon", "wt_codon_rarely_used", "alt_codon_rarely_used",
             "wt_RSCU_gene", "alt_RSCU_gene", "dRSCU_gene", "wt_wCAI", "alt_wCAI", "delta_cai_weight",
             "wt_wtAI", "alt_wtAI", "delta_tai_weight", "dGens_codon_40", "SpliceAI_DS_max",
             "primates_with_alt_codon", "species_with_alt_codon", "evolutionary_class"]
unresolved = esm1["in_clinvar"].fillna(False).astype(bool) & \
             esm1["clinsig"].str.contains("Uncertain|Conflicting", case=False, na=False)
esm2 = esm1.loc[unresolved, ESM2_COLS].reset_index(drop=True)
n_vus = int(esm2.clinsig.str.contains("Uncertain").sum())
n_conf = int(esm2.clinsig.str.contains("Conflicting").sum())
print(f"Unresolved ClinVar variants: {len(esm2)} ({n_vus} uncertain significance, {n_conf} conflicting); "
      f"rarely used reference codon: {int(esm2.wt_codon_rarely_used.sum())}; "
      f"rarely used alternative codon: {int(esm2.alt_codon_rarely_used.sum())}")
write_esm(esm2, os.path.join(WORKDIR, "ESM_2.xlsx"), 2,
          "Synonymous variants reported in ClinVar with uncertain or conflicting classifications, "
          "with their codon usage, tRNA adaptation and local folding energy changes and maximum SpliceAI delta score.")

Saved ESM_1.xlsx: 1130 rows x 55 columns
Unresolved ClinVar variants: 21 (8 uncertain significance, 13 conflicting); rarely used reference codon: 8; rarely used alternative codon: 5
Saved ESM_2.xlsx: 21 rows x 27 columns


In [5]:
# == Cell Z == Download the results =================================================
shutil.make_archive("spliceai_results", "zip", WORKDIR)
try:
    from google.colab import files
    files.download("spliceai_results.zip")
except ImportError:
    print("Results archived in spliceai_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>